# Desarrollo y evaluación de agentes de Mario Bros

Este notebook es el informe técnico y experimental del proyecto. Consolida los entrenamientos ejecutados con DQN y PPO. Las cifras se cargan desde los CSV producidos por `evaluate.py`; no se escriben manualmente.

**Reproducibilidad:** ejecutar desde la raíz del repositorio después de generar los resultados. Si una carpeta no contiene CSV, las celdas de análisis muestran un aviso y continúan sin inventar datos.

## 1. Contexto, usuarios y decisión apoyada

El problema es el *playtesting* automático de un videojuego: entrenar un agente que observe el estado visual de Super Mario Bros., seleccione acciones discretas y avance por un nivel. Los usuarios son estudiantes, docentes, investigadores y diseñadores de niveles que necesitan comparar agentes, estudiar su comportamiento o detectar zonas de dificultad y bloqueo.

El modelo apoya la decisión de qué acción ejecutar en cada instante (`NOOP`, avanzar, saltar o combinaciones de movimiento) para maximizar el progreso y la probabilidad de alcanzar la bandera. No sustituye el criterio del usuario: la aplicación permite auditar partidas, revisar métricas y comparar checkpoints.

## 2. Datos generados y preprocesamiento

Este es un proyecto de aprendizaje por refuerzo, por lo que no parte de un dataset estático ni de etiquetas humanas. Las observaciones y recompensas se generan durante la interacción con el entorno emulado por `gym-super-mario-bros`, apoyado en `nes-py`.

- Los frames RGB del juego se convierten a escala de grises y se redimensionan a `84x84`.
- Se apilan cuatro frames consecutivos: la política recibe una observación `(4, 84, 84)` `uint8`.
- Se repite cada acción durante cuatro frames (`frame skip = 4`) y se suman las recompensas.
- La recompensa combina la señal del entorno con el *shaping* de bandera, potenciadores y pérdida de poder.
- `x_pos`, `flag_get` y `status` se usan para recompensas o métricas; no se entregan como entradas a la política.

Por ello, el análisis exploratorio equivalente al EDA se realiza sobre trayectorias y episodios: recompensas, duración, progreso, posiciones de muerte y completitud.

## 3. Diseño experimental y decisiones

### Entorno

El contrato congelado de `env-v2` usa las siete acciones de `SIMPLE_MOVEMENT`, `frame skip = 4`, escala de grises, resolución `84x84`, cuatro frames apilados y `noop_max = 30` al reiniciar. El nivel `1-1` es el entorno de entrenamiento; `2-1` y `3-1` se reservan como niveles no vistos. Las longitudes de referencia se consultan en `levels.json`.

### Modelos

- **DQN (modelo principal):** se eligió porque el espacio de acciones es discreto. El replay buffer rompe correlaciones entre transiciones, la red objetivo estabiliza los valores Q y epsilon-greedy permite explorar.
- **PPO (baseline):** ofrece una comparación con una política actor-crítico que usa clipping y GAE. No se presenta como un conjunto de datos diferente, sino como otro algoritmo bajo el mismo contrato de observación y evaluación.

### Variantes de ejecución y entrenamiento

Los notebooks `colab_train.ipynb` y `kaggle_train.ipynb` son lanzadores de infraestructura: ejecutan los mismos scripts del repositorio, pero aprovechan recursos y almacenamiento distintos. Colab es el flujo principal con GPU y Drive persistente; Kaggle es una alternativa de cómputo, principalmente para PPO. `transfer_learning_mario.ipynb` sí es una variante experimental: inicializa la CNN de DQN con características visuales de PPO, congela la CNN en una fase inicial y después hace *fine-tuning*. La comparación relevante es entrenamiento desde cero frente a transfer learning, no Colab frente a Kaggle.

## 4. Paso a paso del entrenamiento

1. Se verifica el entorno con `scripts/smoke_test_env.py` y se comprueba la forma, tipo y rango de las observaciones.
2. Se fija la semilla, el nivel, el presupuesto de pasos, `noop_max`, el dispositivo y las rutas de checkpoints/logs.
3. Se ejecuta `dqn.train` o `ppo.train`. DQN almacena transiciones en el replay buffer y actualiza la red según sus hiperparámetros; PPO recolecta rollouts y actualiza su política con clipping y GAE.
4. Se guardan checkpoints y logs de TensorBoard. En Colab se sincronizan con Drive; en Kaggle se descargan desde los artefactos de la sesión.
5. Se selecciona el checkpoint usando el conjunto de validación, sin utilizar las semillas de evaluación final.
6. Se ejecuta `evaluate.py` con un episodio por semilla y se generan CSV/JSON. La evaluación final usa semillas distintas y reporta progreso, completitud, recompensa, pasos y posición de muerte.

La separación entre entrenamiento, validación y evaluación final evita elegir el modelo con base en los mismos episodios usados para reportar generalización.

## 5. Protocolo de evaluación

`evaluate.py` usa el mismo procedimiento para todos los agentes: un episodio por semilla, `noop_max = 30`, progreso `max_x / level_length` y `completed = flag_get`. Para DQN se reporta epsilon-greedy fijo de `0.05`; PPO muestrea su política. El modo determinista se reporta como resultado secundario porque epsilon-greedy y muestreo de PPO no son aleatoriedades equivalentes.

- Validación: semillas `100–104`, principalmente sobre `1-1`, para elegir checkpoint.
- Evaluación final: semillas `200–209` y, cuando existan resultados, niveles no vistos `2-1` y `3-1`.
- Métrica principal: progreso medio y su dispersión.
- Métricas secundarias: tasa de completitud, recompensa, pasos y posición de muerte.

No se afirma que un agente completó el juego si los resultados solo muestran progreso parcial.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path("results")
def load_results(pattern="**/*.csv"):
    files = sorted(RESULTS.glob(pattern))
    frames = []
    for path in files:
        frame = pd.read_csv(path)
        frame["source_file"] = path.name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

df = load_results()
if df.empty:
    print("No hay CSV en results/. Ejecuta evaluate.py antes de analizar resultados.")
else:
    print(f"{len(df)} episodios cargados desde results/")
    display(df.head())

## 6. Análisis exploratorio de trayectorias

En RL, estas distribuciones cumplen el papel del EDA: permiten detectar inestabilidad, recompensas anómalas, episodios truncados y zonas donde el agente falla.

In [ ]:
required = {"algo", "level", "completed", "max_x", "steps", "reward", "progress"}
if not df.empty and required.issubset(df.columns):
    summary = (df.groupby(["algo", "level", "prediction_mode"], dropna=False)
                .agg(episodes=("completed", "size"),
                     completion_rate=("completed", "mean"),
                     progress_mean=("progress", "mean"),
                     progress_std=("progress", "std"),
                     reward_mean=("reward", "mean"),
                     steps_mean=("steps", "mean"),
                     death_x_mean=("death_x", "mean"))
                .round(3))
    display(summary)
else:
    print("Faltan CSV o columnas necesarias para construir el resumen.")

In [ ]:
if not df.empty and {"algo", "level", "progress"}.issubset(df.columns):
    ax = df.boxplot(column="progress", by=["algo", "level"], rot=30, figsize=(10, 5))
    plt.suptitle("")
    ax.set_title("Distribución del progreso por algoritmo y nivel")
    ax.set_ylabel("Progreso normalizado")
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos suficientes para graficar progreso.")

## 7. Insights y comparación de experimentos

Completar esta sección después de ejecutar las celdas con los CSV definitivos. Los insights deben ser observaciones verificables, no expectativas:

- Comparar DQN y PPO bajo el mismo nivel, semillas y protocolo.
- Comparar entrenamiento desde cero con transfer learning usando el mismo presupuesto de pasos cuando sea posible.
- Identificar si una mayor recompensa se corresponde con mayor progreso y completitud; el *reward shaping* puede hacer que no sean equivalentes.
- Reportar la variabilidad entre semillas y las posiciones de muerte, no solo el promedio.
- Separar el desempeño en `1-1` (visto durante entrenamiento) del desempeño en `2-1` y `3-1` (no vistos).

Una conclusión debe incluir la métrica, el subconjunto de datos y la limitación que la acompaña. Por ejemplo: el mejor promedio de progreso en validación no demuestra por sí solo generalización a niveles no vistos.

## 8. Generalización, limitaciones y despliegue

La generalización se estima evaluando semillas no utilizadas y niveles no vistos. El entorno emulado no representa jugadores reales, las semillas no cubren toda la variabilidad posible y DQN/PPO tienen mecanismos de exploración distintos. Además, el resultado depende de la versión congelada de `env-v2`, el presupuesto de pasos y la configuración del hardware.

El modelo se despliega para usuarios mediante `app/main.py` (Streamlit), que permite seleccionar agentes, revisar resultados, reproducir videos y auditar métricas. Si se reentrena un agente, el procedimiento recomendado es: evaluar en el mismo protocolo, comparar con la versión anterior y conservar la versión anterior si el desempeño o la generalización empeoran.